# DM Test Tables: Table 2, 8, 9, 10, 11

Reproduces the Diebold-Mariano test tables (Win-Tie-Loss summary, standard, and HAC-robust) directly from per-model prediction files under `./result/pred`, and writes every output straight into `./result/`, named after the paper's own Table numbers.

## Required inputs
- `result/date.npy` - shared trading-date index (SPY/DIA/QQQ share the same NYSE calendar).
- `result/pred/Seed{1..5}_{model}_{symbol}_results.npy` - 5 seeds x 16 models x 3 symbols = 240 files.
- `result/target_{symbol}_results.npy` - ground-truth realized variance, 3 files (reused across all 5 seeds).

## Outputs (this notebook)
| File | Maps to |
|---|---|
| `result/Table2.csv` | Table 2 - Win-Tie-Loss summary at 5% significance (non-HAC) |
| `result/Table8.csv` | Table 8 - DM test statistics (per-seed, non-HAC) |
| `result/Table9.csv` | Table 9 - DM test p-values (per-seed, non-HAC) |
| `result/Table10.csv` | Table 10 - HAC-corrected DM test statistics (per-seed) |
| `result/Table11.csv` | Table 11 - HAC-corrected DM test p-values (per-seed) |

HAC = Newey-West (Bartlett kernel, automatic bandwidth). Both DM variants use the same right-tailed one-sided test ($H_0: E[d_t] \le 0$); positive DM = DeepHAR outperforms the benchmark.

In [1]:
import os

os.makedirs('result', exist_ok=True)

In [2]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
from sklearn import preprocessing
import statsmodels.api as sm
from scipy.special import gamma
from scipy.stats import norm
import copy
from scipy.stats import kurtosis, skew

def MSE(prediction_samples, labels) :
    N = labels.shape[0]
    prediction_samples = prediction_samples.reshape(-1)
    labels = labels.reshape(-1)
    return np.sum(np.square(prediction_samples - labels)) / N

def QLIKE(prediction_samples, labels) :
    N = labels.shape[0]
    prediction_samples = prediction_samples.reshape(-1)
    labels = labels.reshape(-1)
    return np.sum(labels/ prediction_samples -  np.log(labels/ prediction_samples) - 1)/ N

model_all = ['DeepHAR']
model_all += ['HAR_RV', 'HAR_RV_J', 'HAR_RV_CJ']
model_all += ['EWMA', 'GARCH', 'GJR-GARCH', 'RGARCH']
model_all += ['LSTM', 'GRU', 'BILSTM']
model_all += ['DLinear', 'PatchTST', 'iTransformer', 'TimeMixer', 'TimeXer']

model_order = [ 'HAR_RV', 'HAR_RV_J', 'HAR_RV_CJ', 'EWMA', 'GARCH', 'GJR-GARCH', 'RGARCH', 'LSTM', 'GRU', 'BILSTM',
    'DLinear', 'PatchTST', 'iTransformer', 'TimeMixer', 'TimeXer', 'DeepHAR',
]
symbols = ['SPY', 'DIA', 'QQQ']

In [3]:
# ============================================================
# Load all model predictions from ./result/pred (+ target from ./result)
# -> feeds Table 8, 9, 10, 11 below
# ============================================================
seed_list = [1, 2, 3, 4, 5]   # files are saved as Seed1_ ~ Seed5_

import os
results = pd.DataFrame() 
results_list = []
for random_seed in seed_list:
    for symbol in symbols :   
        date_index = np.load(f'result/date.npy')
        for model in  model_all:
            results_i = pd.DataFrame()
            output_path = os.path.join(f'./result/pred/Seed{random_seed}_' + model + '_' + symbol  + '_results.npy')
            values = np.load(output_path)
            results_i.loc[:, 'pred'] = values.reshape(-1)
            results_i.loc[:, 'date'] = date_index
            results_i.loc[:, 'symbol'] = symbol
            results_i.loc[:, 'model'] = model
            results_i.loc[:, 'random_seed'] = random_seed
            results_list.append(results_i)
    
        results_i = pd.DataFrame()
        output_path = os.path.join(f'./result/target' + '_' + symbol  + '_results.npy')
        values = np.load(output_path)
        results_i.loc[:, 'pred'] = values.reshape(-1)
        results_i.loc[:, 'date'] = date_index
        results_i.loc[:, 'symbol'] = symbol
        results_i.loc[:, 'model'] = 'target'
        results_i.loc[:, 'random_seed'] = random_seed
        results_list.append(results_i)

results = pd.concat(results_list, axis =0)
print(f"Loaded {len(results)} rows from ./result/pred (models x seeds x symbols) + target")

Loaded 384540 rows from ./result/pred (models x seeds x symbols) + target


In [4]:
import collections
from scipy.stats import norm

def hac_variance(d, lag=None):
    """Newey-West (Bartlett kernel) long-run variance estimator."""
    d = np.asarray(d)
    T = len(d)
    d_demean = d - d.mean()

    if lag is None:
        # Newey & West (1994) automatic bandwidth rule of thumb
        lag = int(np.floor(4 * (T / 100) ** (2 / 9)))

    gamma_0 = np.sum(d_demean ** 2) / T
    var_hac = gamma_0
    for k in range(1, lag + 1):
        gamma_k = np.sum(d_demean[k:] * d_demean[:-k]) / T
        w = 1 - k / (lag + 1)          # Bartlett weight
        var_hac += 2 * w * gamma_k

    return var_hac, lag


def dm_test(target, model_val, bench_val, h=1, crit="MSE", hac=False, lag=None):
    T = float(len(target))

    if crit == "MSE":
        e1_lst = np.square(target - model_val)
        e2_lst = np.square(target - bench_val)
    elif crit == "QLIKE":
        e1_lst = (target / model_val) - np.log(target / model_val) - 1
        e2_lst = (target / bench_val) - np.log(target / bench_val) - 1

    d_lst = e2_lst - e1_lst
    mean_d = np.mean(d_lst)

    if hac:
        var_d, lag_used = hac_variance(d_lst, lag=lag)
    else:
        var_d, lag_used = np.var(d_lst, ddof=0), 0

    V_d = np.sqrt(var_d / T)
    DM_stat = mean_d / V_d
    p_value = 1 - norm.cdf(DM_stat)

    dm_return = collections.namedtuple('dm_return', 'DM p_value lag')
    return dm_return(DM=DM_stat, p_value=p_value, lag=lag_used)


def get_stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.1: return "*"
    return ""

## Table 8, Table 9 - Diebold-Mariano test (standard, non-HAC)

In [5]:
# 설정
target_model = "DeepHAR"
seeds = seed_list  # [1, 2, 3, 4, 5]
analysis_targets = symbols

summary_list = []  # Win-Tie-Loss 집계용
detail_list = []   # Table 8 (DM statistics)
pval_list = []     # Table 9 (p-values)
MSE_win = 0
QLIKE_win = 0

for target in analysis_targets:
    # 시장 데이터 필터링
    if target == 'ALL':
        target_df = results.copy()
    else:
        target_df = results[results['symbol'] == target]

    # 비교 대상 모델들
    comparison_models = [m for m in model_order if m in target_df['model'].unique() and m != target_model]

    for comp_model in comparison_models:
        # W-T-L 카운터 (DeepHAR 기준)
        mse_stats = {'W': 0, 'T': 0, 'L': 0}
        qlike_stats = {'W': 0, 'T': 0, 'L': 0}

        seed_pvals = {}
        pval = {}
        for n, seed in enumerate(seeds):
            seed_df = target_df[target_df['random_seed'] == seed]
            # 정렬 보장을 위한 피벗
            pivot = seed_df.pivot_table(index=['date', 'symbol'], columns='model', values='pred').dropna()

            if target_model not in pivot.columns or comp_model not in pivot.columns:
                continue

            # DM Test 수행 (non-HAC)
            dm_mse = dm_test(pivot['target'], pivot[target_model], pivot[comp_model], crit="MSE")
            dm_qlike = dm_test(pivot['target'], pivot[target_model], pivot[comp_model], crit="QLIKE")

            # MSE 판별
            if dm_mse.p_value < 0.05:
                if dm_mse.DM > 0: mse_stats['W'] += 1
                else: mse_stats['L'] += 1
            else: mse_stats['T'] += 1

            if dm_mse.p_value < 0.1: MSE_win += 1
            if dm_qlike.p_value < 0.1: QLIKE_win += 1

            # QLIKE 판별
            if dm_qlike.p_value < 0.05:
                if dm_qlike.DM > 0: qlike_stats['W'] += 1
                else: qlike_stats['L'] += 1
            else: qlike_stats['T'] += 1

            # Table 8: DM statistic + stars
            seed_pvals[f'Seed{n+1}_M'] = f"{dm_mse.DM:.3f}{get_stars(dm_mse.p_value)}"
            seed_pvals[f'Seed{n+1}_Q'] = f"{dm_qlike.DM:.3f}{get_stars(dm_qlike.p_value)}"
            # Table 9: p-value + stars
            pval[f'Seed{n+1}_M'] = f"{dm_mse.p_value:.3f}{get_stars(dm_mse.p_value)}"
            pval[f'Seed{n+1}_Q'] = f"{dm_qlike.p_value:.3f}{get_stars(dm_qlike.p_value)}"

        summary_list.append({
            'Market': target,
            'Model': comp_model,
            'MSE (W-T-L)': f"{mse_stats['W']}-{mse_stats['T']}-{mse_stats['L']}",
            'QLIKE (W-T-L)': f"{qlike_stats['W']}-{qlike_stats['T']}-{qlike_stats['L']}"
        })

        detail_entry = {'Market': target, 'Model': comp_model}
        pval_entry = {'Market': target, 'Model': comp_model}
        detail_entry.update(seed_pvals)
        pval_entry.update(pval)
        detail_list.append(detail_entry)
        pval_list.append(pval_entry)

print(f"MSE_win (p<0.1): {MSE_win}, QLIKE_win (p<0.1): {QLIKE_win}")

MSE_win (p<0.1): 190, QLIKE_win (p<0.1): 205


## Table 2 - Win-Tie-Loss summary (DeepHAR vs. benchmarks, non-HAC)

In [6]:
# Table 2: W-T-L 요약 표 생성 (5% 유의수준, non-HAC DM test 기준)
df_summary = pd.DataFrame(summary_list)
final_summary = df_summary.pivot(index='Model', columns='Market', values=['MSE (W-T-L)', 'QLIKE (W-T-L)'])

# 모델 순서 정렬 (행)
existing_models = [m for m in model_order if m in final_summary.index]
final_summary = final_summary.reindex(index=existing_models)

# Market과 Metric 레벨 교체 (SPY가 위로 오게)
final_summary = final_summary.swaplevel(0, 1, axis=1)

# Market 순서 강제 지정 (SPY-DIA-QQQ)
target_symbols = ['SPY', 'DIA', 'QQQ']
final_summary = final_summary.reindex(columns=target_symbols, level=0)

# Metric 순서 지정 (MSE, QLIKE)
target_metrics = ['MSE (W-T-L)', 'QLIKE (W-T-L)']
final_summary = final_summary.reindex(columns=target_metrics, level=1)

final_summary.to_csv("result/Table2.csv")
print(final_summary)

Market               SPY                       DIA                       QQQ  \
             MSE (W-T-L) QLIKE (W-T-L) MSE (W-T-L) QLIKE (W-T-L) MSE (W-T-L)   
Model                                                                          
HAR_RV             3-2-0         5-0-0       3-2-0         5-0-0       2-3-0   
HAR_RV_J           1-4-0         5-0-0       0-5-0         5-0-0       2-3-0   
HAR_RV_CJ          3-2-0         5-0-0       3-2-0         5-0-0       2-3-0   
EWMA               5-0-0         5-0-0       5-0-0         5-0-0       5-0-0   
GARCH              5-0-0         5-0-0       5-0-0         5-0-0       5-0-0   
GJR-GARCH          5-0-0         5-0-0       5-0-0         5-0-0       5-0-0   
RGARCH             4-1-0         5-0-0       5-0-0         5-0-0       5-0-0   
LSTM               5-0-0         3-2-0       5-0-0         4-1-0       5-0-0   
GRU                4-1-0         4-1-0       5-0-0         4-1-0       4-1-0   
BILSTM             3-2-0         4-1-0  

In [7]:
df_detail = pd.DataFrame(detail_list)
df_detail_pval = pd.DataFrame(pval_list)

In [8]:
df_detail.to_csv("result/Table8.csv")   # DM test statistics (non-HAC, per-seed)
print(df_detail)

   Market         Model   Seed1_M   Seed1_Q   Seed2_M   Seed2_Q   Seed3_M  \
0     SPY        HAR_RV   1.699**  3.775***   1.843**  3.149***    1.484*   
1     SPY      HAR_RV_J    1.582*  2.857***   1.819**  2.542***    1.381*   
2     SPY     HAR_RV_CJ   1.707**  4.118***   1.783**  3.506***    1.487*   
3     SPY          EWMA  3.744***  8.270***  3.403***  8.167***  3.626***   
4     SPY         GARCH  3.661***  7.250***  3.308***  7.034***  3.515***   
5     SPY     GJR-GARCH  2.813***  7.189***  2.390***  6.871***  2.722***   
6     SPY        RGARCH   1.778**  5.045***   1.725**  4.875***   1.686**   
7     SPY          LSTM  2.376***  2.832***   1.760**     0.533  2.338***   
8     SPY           GRU   2.187**  2.622***   1.697**   2.101**  2.383***   
9     SPY        BILSTM  2.729***  4.015***   1.754**     1.004  2.356***   
10    SPY       DLinear     0.800  4.031***     0.474  3.278***     0.421   
11    SPY      PatchTST  3.567***  7.568***  3.310***  5.120***   2.233**   

In [9]:
df_detail_pval.to_csv("result/Table9.csv")   # DM test p-values (non-HAC, per-seed)
print(df_detail_pval)

   Market         Model   Seed1_M   Seed1_Q   Seed2_M   Seed2_Q   Seed3_M  \
0     SPY        HAR_RV   0.045**  0.000***   0.033**  0.001***    0.069*   
1     SPY      HAR_RV_J    0.057*  0.002***   0.034**  0.006***    0.084*   
2     SPY     HAR_RV_CJ   0.044**  0.000***   0.037**  0.000***    0.069*   
3     SPY          EWMA  0.000***  0.000***  0.000***  0.000***  0.000***   
4     SPY         GARCH  0.000***  0.000***  0.000***  0.000***  0.000***   
5     SPY     GJR-GARCH  0.002***  0.000***  0.008***  0.000***  0.003***   
6     SPY        RGARCH   0.038**  0.000***   0.042**  0.000***   0.046**   
7     SPY          LSTM  0.009***  0.002***   0.039**     0.297  0.010***   
8     SPY           GRU   0.014**  0.004***   0.045**   0.018**  0.009***   
9     SPY        BILSTM  0.003***  0.000***   0.040**     0.158  0.009***   
10    SPY       DLinear     0.212  0.000***     0.318  0.001***     0.337   
11    SPY      PatchTST  0.000***  0.000***  0.000***  0.000***   0.013**   

## Table 10, Table 11 - HAC-corrected Diebold-Mariano test (Newey-West)

In [10]:
# 설정 (HAC 버전)
target_model = "DeepHAR"
seeds = seed_list  # [1, 2, 3, 4, 5]
analysis_targets = symbols

summary_list = []  # Win-Tie-Loss 집계용
detail_list = []   # Table 10 (HAC DM statistics)
pval_list = []     # Table 11 (HAC p-values)
MSE_win = 0
QLIKE_win = 0

for target in analysis_targets:
    if target == 'ALL':
        target_df = results.copy()
    else:
        target_df = results[results['symbol'] == target]

    comparison_models = [m for m in model_order if m in target_df['model'].unique() and m != target_model]

    for comp_model in comparison_models:
        mse_stats = {'W': 0, 'T': 0, 'L': 0}
        qlike_stats = {'W': 0, 'T': 0, 'L': 0}

        seed_pvals = {}
        pval = {}
        for n, seed in enumerate(seeds):
            seed_df = target_df[target_df['random_seed'] == seed]
            pivot = seed_df.pivot_table(index=['date', 'symbol'], columns='model', values='pred').dropna()

            if target_model not in pivot.columns or comp_model not in pivot.columns:
                continue

            # DM Test 수행 (HAC-robust)
            dm_mse = dm_test(pivot['target'], pivot[target_model], pivot[comp_model], crit="MSE", hac=True)
            dm_qlike = dm_test(pivot['target'], pivot[target_model], pivot[comp_model], crit="QLIKE", hac=True)

            if dm_mse.p_value < 0.05:
                if dm_mse.DM > 0: mse_stats['W'] += 1
                else: mse_stats['L'] += 1
            else: mse_stats['T'] += 1

            if dm_mse.p_value < 0.1: MSE_win += 1
            if dm_qlike.p_value < 0.1: QLIKE_win += 1

            if dm_qlike.p_value < 0.05:
                if dm_qlike.DM > 0: qlike_stats['W'] += 1
                else: qlike_stats['L'] += 1
            else: qlike_stats['T'] += 1

            # Table 10: HAC DM statistic + stars
            seed_pvals[f'Seed{n+1}_M'] = f"{dm_mse.DM:.3f}{get_stars(dm_mse.p_value)}"
            seed_pvals[f'Seed{n+1}_Q'] = f"{dm_qlike.DM:.3f}{get_stars(dm_qlike.p_value)}"
            # Table 11: HAC p-value + stars
            pval[f'Seed{n+1}_M'] = f"{dm_mse.p_value:.3f}{get_stars(dm_mse.p_value)}"
            pval[f'Seed{n+1}_Q'] = f"{dm_qlike.p_value:.3f}{get_stars(dm_qlike.p_value)}"

        summary_list.append({
            'Market': target,
            'Model': comp_model,
            'MSE (W-T-L)': f"{mse_stats['W']}-{mse_stats['T']}-{mse_stats['L']}",
            'QLIKE (W-T-L)': f"{qlike_stats['W']}-{qlike_stats['T']}-{qlike_stats['L']}"
        })

        detail_entry = {'Market': target, 'Model': comp_model}
        pval_entry = {'Market': target, 'Model': comp_model}
        detail_entry.update(seed_pvals)
        pval_entry.update(pval)
        detail_list.append(detail_entry)
        pval_list.append(pval_entry)

print(f"HAC MSE_win (p<0.1): {MSE_win}, HAC QLIKE_win (p<0.1): {QLIKE_win}")

HAC MSE_win (p<0.1): 134, HAC QLIKE_win (p<0.1): 195


In [11]:
df_detail = pd.DataFrame(detail_list)
df_detail_pval = pd.DataFrame(pval_list)

In [12]:
df_detail.to_csv("result/Table10.csv")   # DM test statistics (HAC-robust, per-seed)
print(df_detail)

   Market         Model  Seed1_M   Seed1_Q  Seed2_M   Seed2_Q  Seed3_M  \
0     SPY        HAR_RV   1.390*  3.464***   1.588*  2.892***   1.318*   
1     SPY      HAR_RV_J    1.248   2.289**   1.505*   2.105**    1.213   
2     SPY     HAR_RV_CJ   1.405*  3.455***  1.651**  2.949***   1.339*   
3     SPY          EWMA  1.810**  5.132***   1.640*  4.972***  1.758**   
4     SPY         GARCH  1.770**  4.912***   1.599*  4.730***  1.712**   
5     SPY     GJR-GARCH   1.523*  4.835***    1.270  4.618***   1.438*   
6     SPY        RGARCH    1.238  3.641***    1.232  3.541***    1.205   
7     SPY          LSTM   1.376*  2.520***    1.040     0.430   1.306*   
8     SPY           GRU   1.367*  2.653***    0.996   1.838**   1.282*   
9     SPY        BILSTM   1.526*  3.365***    1.021     0.839   1.358*   
10    SPY       DLinear    0.630  3.932***    0.419  3.097***    0.387   
11    SPY      PatchTST  1.738**  5.088***  2.028**  3.986***  1.965**   
12    SPY  iTransformer   1.441*  2.37

In [13]:
df_detail_pval.to_csv("result/Table11.csv")   # DM test p-values (HAC-robust, per-seed)
print(df_detail_pval)

   Market         Model  Seed1_M   Seed1_Q  Seed2_M   Seed2_Q  Seed3_M  \
0     SPY        HAR_RV   0.082*  0.000***   0.056*  0.002***   0.094*   
1     SPY      HAR_RV_J    0.106   0.011**   0.066*   0.018**    0.112   
2     SPY     HAR_RV_CJ   0.080*  0.000***  0.049**  0.002***   0.090*   
3     SPY          EWMA  0.035**  0.000***   0.050*  0.000***  0.039**   
4     SPY         GARCH  0.038**  0.000***   0.055*  0.000***  0.043**   
5     SPY     GJR-GARCH   0.064*  0.000***    0.102  0.000***   0.075*   
6     SPY        RGARCH    0.108  0.000***    0.109  0.000***    0.114   
7     SPY          LSTM   0.084*  0.006***    0.149     0.334   0.096*   
8     SPY           GRU   0.086*  0.004***    0.160   0.033**   0.100*   
9     SPY        BILSTM   0.064*  0.000***    0.154     0.201   0.087*   
10    SPY       DLinear    0.264  0.000***    0.337  0.001***    0.349   
11    SPY      PatchTST  0.041**  0.000***  0.021**  0.000***  0.025**   
12    SPY  iTransformer   0.075*  0.00